# Chapter 2 Exercise solutions

## Exercise 2.1

In [1]:
import tiktoken

tokenizer = tiktoken.get_encoding(encoding_name="gpt2")

In [9]:
text = "Akwirw ier"

integers = tokenizer.encode(text)
print(integers)

[33901, 86, 343, 86, 220, 959]


In [29]:
for i in integers:
    print(f"{i:<6} -> {tokenizer.decode([i]):>6}")

33901  ->     Ak
86     ->      w
343    ->     ir
86     ->      w
220    ->       
959    ->    ier


In [24]:
tokenizer.decode(integers)

'Akwirw ier'

## Exercise 2.2

In [33]:
from pathlib import Path

import torch
from torch.utils.data import Dataset, DataLoader

In [35]:
file_path = Path("the-verdict.txt")

In [ ]:
class GPTDatasetV1(Dataset):
    def __init__(
        self, txt: str, tokenizer: tiktoken.Encoding, max_length: int, stride: int
    ) -> None:
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        assert (
            len(token_ids) > max_length
        ), "Number of tokenized inputs must at least be equal to max_length + 1"

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i : i + max_length]
            target_chunk = token_ids[i + 1 : i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self) -> int:
        return len(self.input_ids)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        return self.input_ids[index], self.target_ids[index]

In [43]:
def create_dataloader_v1(
    txt: str,
    batch_size: int = 4,
    max_length: int = 256,
    stride: int = 128,
    shuffle=True,
    drop_last=True,
    num_workers=0,
) -> DataLoader:
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers,
    )

In [44]:
with open(file_path, "r", encoding="utf-8") as f:
    raw_text = f.read()

In [45]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=3, max_length=2, stride=2, shuffle=False
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print(f"Inputs:\n{inputs}")
print(f"\nTargets:\n{targets}")

Inputs:
tensor([[  40,  367],
        [2885, 1464],
        [1807, 3619]])

Targets:
tensor([[ 367, 2885],
        [1464, 1807],
        [3619,  402]])


In [46]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=3, max_length=8, stride=2, shuffle=False
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print(f"Inputs:\n{inputs}")
print(f"\nTargets:\n{targets}")

Inputs:
tensor([[   40,   367,  2885,  1464,  1807,  3619,   402,   271],
        [ 2885,  1464,  1807,  3619,   402,   271, 10899,  2138],
        [ 1807,  3619,   402,   271, 10899,  2138,   257,  7026]])

Targets:
tensor([[  367,  2885,  1464,  1807,  3619,   402,   271, 10899],
        [ 1464,  1807,  3619,   402,   271, 10899,  2138,   257],
        [ 3619,   402,   271, 10899,  2138,   257,  7026, 15632]])
